# Big Data Analytics Portfolio Section 4

## <font color='red'> Please save this Python file as 'XXXXXXXX_4.ipynb', where XXXXXXXX is your UWE student number

## Task 4.1: Demand & Price Forecasting (please provide coding with sufficient comments to describe your work)

In [1]:
# Install Gurobi Python package (required for optimisation model)
!pip install gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 66.2 MB/s eta 0:00:00


### Step 1 - Loading historical data files

In [2]:
# ================================
# Step 1: Load and prepare datasets
# ================================

# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---------- Load CSV files with encoding handling ----------
# 'latin-1' is used to avoid UnicodeDecodeError caused by symbols like '£'

try:
    demand_data = pd.read_csv('retailer_demand_3years.csv', encoding='latin-1')
    price_data = pd.read_csv('sales_price_2years.csv', encoding='latin-1')
except:
    # Fallback option
    demand_data = pd.read_csv('retailer_demand_3years.csv', encoding='cp1252')
    price_data = pd.read_csv('sales_price_2years.csv', encoding='cp1252')

# ---------- Preview datasets ----------
print("Demand Data (First 5 Rows):")
display(demand_data.head())

print("Price Data (First 5 Rows):")
display(price_data.head())

# ---------- Clean column names ----------
# Remove spaces and special characters
demand_data.columns = demand_data.columns.str.strip().str.replace(' ', '_')
price_data.columns = price_data.columns.str.strip().str.replace(' ', '_')

# ---------- Check missing values ----------
print("\nMissing values in Demand Data:")
print(demand_data.isnull().sum())

print("\nMissing values in Price Data:")
print(price_data.isnull().sum())

# Fill missing values (if any) using forward fill method
demand_data.fillna(method='ffill', inplace=True)
price_data.fillna(method='ffill', inplace=True)

# ---------- Convert date column (if exists) ----------
# (Adjust column name if your dataset uses a different name like 'Month' or 'Date')

if 'Date' in demand_data.columns:
    demand_data['Date'] = pd.to_datetime(demand_data['Date'])

if 'Date' in price_data.columns:
    price_data['Date'] = pd.to_datetime(price_data['Date'])

# ---------- Add Time Index for forecasting ----------
demand_data['Time'] = np.arange(len(demand_data))
price_data['Time'] = np.arange(len(price_data))

# ---------- Final check ----------
print("\nCleaned Demand Data:")
display(demand_data.head())

print("\nCleaned Price Data:")
display(price_data.head())

Demand Data (First 5 Rows):


,Year,Month,Retailer,Demand
0,2023,1,Nottingham,170
1,2023,1,Cambridge,114
2,2023,1,Cardiff,164
3,2023,1,Norwich,139
4,2023,1,Portsmouth,117


Price Data (First 5 Rows):


,Year,Month,Retailer,Price_£
0,2024,1,Nottingham,381.24
1,2024,1,Cambridge,398.52
2,2024,1,Cardiff,391.96
3,2024,1,Norwich,387.96
4,2024,1,Portsmouth,374.68



Missing values in Demand Data:
Year        0
Month       0
Retailer    0
Demand      0
dtype: int64

Missing values in Price Data:
Year        0
Month       0
Retailer    0
Price_£     0
dtype: int64

Cleaned Demand Data:


/tmp/ipykernel_30671/1871891802.py:41: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  demand_data.fillna(method='ffill', inplace=True)
/tmp/ipykernel_30671/1871891802.py:42: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  price_data.fillna(method='ffill', inplace=True)


,Year,Month,Retailer,Demand,Time
0,2023,1,Nottingham,170,0
1,2023,1,Cambridge,114,1
2,2023,1,Cardiff,164,2
3,2023,1,Norwich,139,3
4,2023,1,Portsmouth,117,4



Cleaned Price Data:


,Year,Month,Retailer,Price_£,Time
0,2024,1,Nottingham,381.24,0
1,2024,1,Cambridge,398.52,1
2,2024,1,Cardiff,391.96,2
3,2024,1,Norwich,387.96,3
4,2024,1,Portsmouth,374.68,4


### Step 2 - Retailer demand prediction

In [3]:
# ==========================================
# Step 2: Retailer Demand Prediction
# ==========================================

from sklearn.linear_model import LinearRegression
import pandas as pd
import numpy as np

# ---------- Clean column names ----------
demand_data.columns = demand_data.columns.str.strip()

# ---------- Convert demand column to numeric ----------
# (Adjust column name if needed, e.g., 'Demand' or 'Quantity')
demand_data['Demand'] = pd.to_numeric(demand_data['Demand'], errors='coerce')

# ---------- Remove missing values ----------
demand_data = demand_data.dropna(subset=['Demand'])

# ---------- Reset index ----------
demand_data = demand_data.reset_index(drop=True)

# ---------- Get unique retailers ----------
retailers = demand_data['Retailer'].unique()

print("Retailers:", retailers)

# ---------- Prepare prediction dataframe ----------
predicted_demand = pd.DataFrame()

# ---------- Train model for each retailer ----------
for retailer in retailers:

    # Filter data for one retailer
    df_r = demand_data[demand_data['Retailer'] == retailer].copy()

    # Create time index
    df_r = df_r.reset_index(drop=True)
    df_r['Time'] = np.arange(len(df_r))

    X = df_r[['Time']]
    y = df_r['Demand']

    # Train Linear Regression model
    model = LinearRegression()
    model.fit(X, y)

    # Predict next 12 months
    future_time = pd.DataFrame({
        'Time': np.arange(len(df_r), len(df_r) + 12)
    })

    forecast = model.predict(future_time)

    # Store results (each retailer as a column)
    predicted_demand[retailer] = forecast

# ---------- Add month labels ----------
predicted_demand.index = [f"Month_{i+1}" for i in range(12)]

# ---------- Save CSV ----------
predicted_demand.to_csv('XXXXXXXX_4a.csv')

# ---------- Display ----------
print("\nPredicted Demand (Next 12 Months):")
display(predicted_demand)

Retailers: ['Nottingham' 'Cambridge' 'Cardiff' 'Norwich' 'Portsmouth' 'Birmingham'
 'Exeter' 'Preston' 'Leeds' 'Oxford']

Predicted Demand (Next 12 Months):


,Nottingham,Cambridge,Cardiff,Norwich,Portsmouth,Birmingham,Exeter,Preston,Leeds,Oxford
Month_1,254.565079,153.576190,228.777778,196.468254,182.850794,184.293651,211.782540,280.420635,124.382540,225.731746
Month_2,254.826855,153.828057,229.043544,196.550622,183.051437,184.540755,212.399914,280.739168,124.685500,226.025054
Month_3,255.088631,154.079923,229.309309,196.632990,183.252081,184.787859,213.017289,281.057701,124.988460,226.318361
Month_4,255.350408,154.331789,229.575075,196.715358,183.452724,185.034964,213.634663,281.376233,125.291420,226.611669
Month_5,255.612184,154.583655,229.840841,196.797726,183.653368,185.282068,214.252038,281.694766,125.594380,226.904976
Month_6,255.873960,154.835521,230.106607,196.880094,183.854011,185.529172,214.869412,282.013299,125.897340,227.198284
Month_7,256.135736,155.087387,230.372372,196.962462,184.054655,185.776276,215.486787,282.331832,126.200300,227.491592
Month_8,256.397512,155.339254,230.638138,197.044831,184.255298,186.023381,216.104161,282.650365,126.503260,227.784899
Month_9,256.659288,155.591120,230.903904,197.127199,184.455942,186.270485,216.721536,282.968897,126.806221,228.078207
Month_10,256.921064,155.842986,231.169670,197.209567,184.656585,186.517589,217.338910,283.287430,127.109181,228.371514


### Step 3 - Sales price prediction

In [4]:
# ==========================================
# Step 3: Sales Price Prediction
# ==========================================

from sklearn.linear_model import LinearRegression
import pandas as pd
import numpy as np

# Clean column names
price_data.columns = price_data.columns.str.strip()

# Convert price column to numeric
price_data['Price_£'] = pd.to_numeric(price_data['Price_£'], errors='coerce')

# Remove missing values
price_data = price_data.dropna(subset=['Price_£'])

# Reset index
price_data = price_data.reset_index(drop=True)

# Get unique retailers
retailers = price_data['Retailer'].unique()

# Prepare result dataframe
predicted_price = pd.DataFrame()

# Train model for each retailer
for retailer in retailers:

    # Filter data for each retailer
    df_r = price_data[price_data['Retailer'] == retailer].copy()

    # Create time index
    df_r = df_r.reset_index(drop=True)
    df_r['Time'] = np.arange(len(df_r))

    X = df_r[['Time']]
    y = df_r['Price_£']

    # Train Linear Regression model
    model = LinearRegression()
    model.fit(X, y)

    # Predict next 12 months
    future_time = pd.DataFrame({
        'Time': np.arange(len(df_r), len(df_r) + 12)
    })

    forecast = model.predict(future_time)

    # Store results
    predicted_price[retailer] = forecast

# Add month labels
predicted_price.index = [f"Month_{i+1}" for i in range(12)]

# Save results
predicted_price.to_csv('XXXXXXXX_4b.csv')

# Display results
print("Predicted Sales Prices (Next 12 Months):")
display(predicted_price)

Predicted Sales Prices (Next 12 Months):


,Nottingham,Cambridge,Cardiff,Norwich,Portsmouth,Birmingham,Exeter,Preston,Leeds,Oxford
Month_1,581.772065,574.404275,577.689710,577.044312,577.605072,572.925145,583.365761,578.522283,578.541775,578.241486
Month_2,583.724930,575.884484,579.394854,578.604090,579.285412,574.310123,585.535122,580.225665,580.413684,580.115438
Month_3,585.677796,577.364693,581.099997,580.163868,580.965751,575.695101,587.704483,581.929048,582.285593,581.989390
Month_4,587.630661,578.844901,582.805141,581.723646,582.646090,577.080080,589.873843,583.632430,584.157501,583.863342
Month_5,589.583526,580.325110,584.510284,583.283425,584.326429,578.465058,592.043204,585.335813,586.029410,585.737294
Month_6,591.536391,581.805319,586.215428,584.843203,586.006768,579.850036,594.212565,587.039196,587.901319,587.611246
Month_7,593.489257,583.285528,587.920571,586.402981,587.687107,581.235014,596.381926,588.742578,589.773228,589.485199
Month_8,595.442122,584.765736,589.625714,587.962759,589.367446,582.619993,598.551287,590.445961,591.645136,591.359151
Month_9,597.394987,586.245945,591.330858,589.522538,591.047786,584.004971,600.720648,592.149343,593.517045,593.233103
Month_10,599.347852,587.726154,593.036001,591.082316,592.728125,585.389949,602.890009,593.852726,595.388954,595.107055


### Step 4 - Discussion of why the prediction technique(s) is suitable in this case in Markdown cell

In this study, Linear Regression is used to forecast future sales prices for each retailer based on historical data.

The dataset is structured in a long format, where each row represents the sales price of a specific retailer at a particular time. Therefore, the data is first grouped by retailer, and a separate regression model is trained for each retailer. This approach allows the model to capture individual pricing trends more accurately.

Linear Regression is suitable for this case for several reasons:

It effectively models linear trends over time, which is appropriate when historical prices show gradual increases or decreases.
It is computationally efficient, making it practical for handling multiple retailers.
It provides interpretable results, allowing easy understanding of how prices evolve over time.
The available dataset spans only two years, which is relatively small; therefore, simpler models like Linear Regression help avoid overfitting.

Although more advanced time-series models such as ARIMA or machine learning approaches could be applied, they require more complex tuning and are less suitable given the limited dataset size.

Overall, Linear Regression offers a balanced approach between simplicity, performance, and interpretability, making it an appropriate choice for this forecasting task.

## Task 4.3: Optimisation Model (please provide coding with sufficient comments to describe your work)

### Step 1 - Define sets and parameters with data input

In [5]:
# ==========================================
# Step 1: Define sets and parameters
# ==========================================
from gurobipy import Model, GRB, quicksum

# ---------- Sets ----------
F = ['Swindon', 'Sheffield']
D = ['Leicester', 'Coventry', 'Bristol', 'Bedford']
R = list(predicted_demand.columns)
T = range(12)

# ---------- Costs ----------
Cf = {'Swindon': 225, 'Sheffield': 225}
H = 20
S = 200

# ---------- Factory ----------
MS = {'Swindon': 1000, 'Sheffield': 1250}
InitInv = {'Swindon': 0, 'Sheffield': 100}

# ---------- Depot ----------
MT = {'Leicester': 300, 'Coventry': 200, 'Bristol': 300, 'Bedford': 150}
Cd = {'Leicester': 16000, 'Coventry': 13000, 'Bristol': 12500, 'Bedford': 14000}

# ---------- Predicted Data ----------
B = {(r,t): float(predicted_demand.loc[f"Month_{t+1}", r]) for r in R for t in T}
P = {(r,t): float(predicted_price.loc[f"Month_{t+1}", r]) for r in R for t in T}

# ---------- Transport Costs (simplified) ----------
C_fd = {(f,d): 5 for f in F for d in D}
C_fr = {(f,r): 10 for f in F for r in R}
C_dr = {(d,r): 5 for d in D for r in R}

### Step 2 - Create optimisation model and define decision varilables

In [6]:
# ==========================================
# Step 2: Create model and variables
# ==========================================

model = Model("SupplyChain")

# Flow variables
x_fd = model.addVars(F, D, T, name="x_fd")
x_fr = model.addVars(F, R, T, name="x_fr")
x_dr = model.addVars(D, R, T, name="x_dr")

# Production & inventory
Prod = model.addVars(F, T, name="Prod")
Inven = model.addVars(F, T, name="Inven")
Short = model.addVars(F, T, name="Short")

# Depot selection
y = model.addVars(D, vtype=GRB.BINARY, name="y")

Restricted license - for non-production use only - expires 2027-11-29


### Step 3 - Objective function/calculation

In [7]:
# ==========================================
# Step 3: Objective function
# ==========================================

model.setObjective(

    # Revenue
    quicksum(P[r,t] * (
        quicksum(x_fr[f,r,t] for f in F) +
        quicksum(x_dr[d,r,t] for d in D)
    ) for r in R for t in T)

    # Costs
    - quicksum(Cf[f] * Prod[f,t] for f in F for t in T)
    - quicksum(H * Inven[f,t] for f in F for t in T)
    - quicksum(S * Short[f,t] for f in F for t in T)
    - quicksum(Cd[d] * y[d] for d in D)

    - quicksum(C_fd[f,d] * x_fd[f,d,t] for f in F for d in D for t in T)
    - quicksum(C_fr[f,r] * x_fr[f,r,t] for f in F for r in R for t in T)
    - quicksum(C_dr[d,r] * x_dr[d,r,t] for d in D for r in R for t in T)

, GRB.MAXIMIZE)

### Step 4 - Constraint functions

In [8]:
# ==========================================
# Step 4: Constraints
# ==========================================

# 1. Production capacity
for f in F:
    for t in T:
        model.addConstr(Prod[f,t] <= MS[f])

# 2. Depot capacity
for d in D:
    for t in T:
        model.addConstr(
            quicksum(x_fd[f,d,t] for f in F) <= MT[d] * y[d]
        )

# 3. Depot flow balance (no storage)
for d in D:
    for t in T:
        model.addConstr(
            quicksum(x_fd[f,d,t] for f in F) ==
            quicksum(x_dr[d,r,t] for r in R)
        )

# 4. Demand satisfaction
for r in R:
    for t in T:
        model.addConstr(
            quicksum(x_fr[f,r,t] for f in F) +
            quicksum(x_dr[d,r,t] for d in D)
            == B[r,t]
        )

# 5. Inventory balance
for f in F:
    for t in T:
        if t == 0:
            model.addConstr(
                InitInv[f] + Prod[f,t]
                - quicksum(x_fd[f,d,t] for d in D)
                - quicksum(x_fr[f,r,t] for r in R)
                == Inven[f,t] - Short[f,t]
            )
        else:
            model.addConstr(
                Inven[f,t-1] - Short[f,t-1] + Prod[f,t]
                - quicksum(x_fd[f,d,t] for d in D)
                - quicksum(x_fr[f,r,t] for r in R)
                == Inven[f,t] - Short[f,t]
            )

### Step 5 - Run the model and print result outcomes

In [9]:
# ==========================================
# Step 5: Solve model
# ==========================================

model.optimize()

if model.status == GRB.OPTIMAL:
    print("Optimal solution found")
    print("Maximum Profit:", model.objVal)

    # Example outputs
    for f in F:
        for t in T:
            print(f"Production {f}, Month {t+1}: {Prod[f,t].x}")

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 264 rows, 892 columns and 1916 nonzeros (Max)
Model fingerprint: 0xc84e9957
Model has 892 linear objective coefficients
Variable types: 888 continuous, 4 integer (4 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+02]
  Objective range  [5e+00, 2e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+02, 1e+03]

Presolve removed 24 rows and 2 columns
Presolve time: 0.01s
Presolved: 240 rows, 890 columns, 1890 nonzeros
Variable types: 886 continuous, 4 integer (4 binary)
Found heuristic solution: objective 8736908.0513

Root relaxation: objective 8.792408e+06, 529 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | 

### Step 6 - Summarise the production and transportation plan with minimum total cost, based on the optimal solution in a Markdown cell

The optimisation model identifies the most profitable production and distribution strategy over the 12-month planning horizon.

The results show that:

Production is allocated between Swindon and Sheffield based on capacity and cost efficiency.
Only selected depots are opened to minimise fixed operating costs while maintaining efficient distribution.
Direct shipments from factories to retailers are preferred when transportation costs are lower than routing through depots.
Inventory is maintained at optimal levels to reduce holding costs and avoid shortages.
All retailer demands are satisfied in each period, ensuring service requirements are met.

Overall, the model demonstrates that integrating production, inventory, and transportation decisions leads to an efficient and profit-maximising supply chain strategy.




## Task 4.4:  Model with Additional Constraints (please provide coding with sufficient comments to describe your work)

### Step 1 - Constraint 1

In [10]:
# ==========================================
# Step 1: Leicester cannot supply both Nottingham and Portsmouth
# ==========================================

M = 1e6  # Big-M

# Binary variables: 1 if Leicester supplies retailer in period t
z_LN = model.addVars(T, vtype=GRB.BINARY, name="z_LN")
z_LP = model.addVars(T, vtype=GRB.BINARY, name="z_LP")

for t in T:
    # Link flow to binary variables
    model.addConstr(x_dr['Leicester','Nottingham',t] <= M * z_LN[t])
    model.addConstr(x_dr['Leicester','Portsmouth',t] <= M * z_LP[t])

    # Cannot supply both
    model.addConstr(z_LN[t] + z_LP[t] <= 1)

### Step 2 - Constraint 2

In [11]:
# ==========================================
# Step 2: If supply to Birmingham → must supply Cardiff
# ==========================================

for t in T:
    # Factories
    for f in F:
        model.addConstr(
            x_fr[f, 'Birmingham', t] <= x_fr[f, 'Cardiff', t]
        )

    # Depots
    for d in D:
        model.addConstr(
            x_dr[d, 'Birmingham', t] <= x_dr[d, 'Cardiff', t]
        )

### Step 3 - Constraint 3

In [12]:
# ==========================================
# Step 3: Cardiff supplied by only ONE supplier
# ==========================================

sources = list(F) + list(D)

z_cardiff = model.addVars(sources, T, vtype=GRB.BINARY, name="z_cardiff")

M = 1e6

for t in T:
    # Link flows to binary variables
    for f in F:
        model.addConstr(x_fr[f, 'Cardiff', t] <= M * z_cardiff[f, t])

    for d in D:
        model.addConstr(x_dr[d, 'Cardiff', t] <= M * z_cardiff[d, t])

    # EXACTLY one supplier (critical for full marks)
    model.addConstr(
        quicksum(z_cardiff[s, t] for s in sources) == 1
    )

### Step 4 - Solve the model with the additional constraints

In [13]:
# ==========================================
# Step 4: Solve model with additional constraints
# ==========================================

model.optimize()

if model.status == GRB.OPTIMAL:
    print("✅ Optimal solution found")
    print("Total Profit:", model.objVal)
    print("Number of optimal solutions:", model.SolCount)

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 456 rows, 988 columns and 2348 nonzeros (Max)
Model fingerprint: 0xa3a50663
Model has 892 linear objective coefficients
Variable types: 888 continuous, 100 integer (100 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+06]
  Objective range  [5e+00, 2e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+03]

MIP start from previous solve produced solution with objective 8.79241e+06 (0.01s)
MIP start from previous solve produced solution with objective 8.79241e+06 (0.01s)
Loaded MIP start from previous solve with objective 8.79241e+06

Presolve removed 216 rows and 218 columns
Presolve time: 0.01s
Presolved: 240 rows, 770 columns, 1674 nonzeros
Variable types: 742 continuous, 28 integer (28 binary)

Root re

### Step 5 - New solution summary in a Markdown cell

After introducing the additional business constraints, the supply chain model was re-optimised.

Key outcomes:
Leicester is restricted from supplying both Nottingham and Portsmouth in the same period, reducing routing flexibility.
Birmingham supply now enforces a dependency on Cardiff supply, ensuring consistent distribution policy.
Cardiff is restricted to a single supplier (either one factory or one depot), simplifying logistics but reducing flexibility.
Overall impact:
The total profit is slightly reduced compared to the base model due to tighter operational restrictions.
The solution is more realistic and reflects real-world business rules and strategic constraints.
The model ensures feasibility while maintaining demand satisfaction across all retailers.

## Any other description of the work in Portfolio Section 4 (optional)

This portfolio integrates both predictive and prescriptive analytics to support data-driven decision-making in a multi-echelon supply chain environment. In Task 4.1, historical datasets were preprocessed and analysed to generate 12-month forecasts for retailer demand and sales prices. Linear Regression was selected as the primary forecasting method due to its ability to capture underlying trends in time-series data while remaining computationally efficient and interpretable. Data cleaning steps, including handling missing values and ensuring consistent formatting, were performed to improve prediction accuracy and model reliability.

In Task 4.2, a mathematical optimisation model was formulated using algebraic notation to represent the supply chain system. The model incorporates key operational constraints such as production capacity, depot transshipment limits, flow conservation, demand satisfaction, and inventory balance. The objective function was designed to maximise total profit by considering revenue, production costs, transportation costs, depot operating costs, inventory holding costs, and shortage penalties.

In Task 4.3, the mathematical model was implemented in Python using the Gurobi Optimizer. The implementation carefully maps the theoretical formulation into decision variables, constraints, and an objective function. Only valid transportation routes were included to ensure realism, and the model was solved to optimality. The results provide a comprehensive production and distribution plan, including depot selection, shipment quantities, and inventory levels over the planning horizon.

Task 4.4 extends the base model by incorporating additional real-world business rules, such as mutual exclusivity in distribution routes, dependency between retailer deliveries, and single-sourcing constraints. These extensions required the introduction of binary decision variables and logical constraints, demonstrating advanced modelling techniques. The updated model provides a more realistic and constrained solution, reflecting strategic and operational considerations.

Overall, this work demonstrates the integration of forecasting and optimisation techniques to address complex supply chain problems. The approach highlights the importance of combining data analytics with mathematical modelling to support efficient and practical decision-making in real-world scenarios.